In [1]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


In [ ]:
# Carregar os dados

# 1. Dados do Cepea 
cepea = pd.read_excel('C:/Users/Microsoft/Desktop/Projects/Python Projects/PrecoBoiBR/data/Boi_gordo_corrigido.xlsx', skiprows=0) 

# 2. Dados do Kaggle
#kaggle = pd.read_csv('data/Boi Gordo Futuros Dados Histricos.csv')

print("Cepea shape:", cepea.shape)
cepea.head()

Cepea shape: (2546, 2)


,Data,Valor
0,04/01/2016,"150,00"
1,05/01/2016,"150,00"
2,06/01/2016,"148,81"
3,07/01/2016,"148,21"
4,08/01/2016,"148,46"


In [ ]:
# Limpeza completa usando as colunas reais: "Data" e "Valor"

# Renomear para facilitar o resto do código
cepea = cepea.rename(columns={'Valor': 'Preco_Arroba'})

# Converter Data para datetime
cepea['Data'] = pd.to_datetime(cepea['Data'], errors='coerce')

# LIMPEZA DO PREÇO 
cepea['Preco_Arroba'] = (
    cepea['Preco_Arroba']
    .astype(str)
    .str.replace(r'R\$', '', regex=True)
    .str.replace(',', '.', regex=False)
    .str.replace('-', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.strip()
)

# Converte para número
cepea['Preco_Arroba'] = pd.to_numeric(cepea['Preco_Arroba'], errors='coerce')

# Remove linhas sem preço
cepea = cepea.dropna(subset=['Preco_Arroba'])

# Ordena por data
cepea = cepea.sort_values('Data').reset_index(drop=True)

print(f"✅ Dados Cepea carregados e limpos: {cepea.shape[0]} linhas")
print(f"Primeira data: {cepea['Data'].iloc[0].date()}")
print(f"Última data: {cepea['Data'].iloc[-1].date()}")
print(f"Último preço: R$ {cepea['Preco_Arroba'].iloc[-1]:.2f}")

cepea.head()

✅ Dados Cepea carregados e limpos: 989 linhas
Primeira data: 2016-01-02
Última data: 2026-12-03
Último preço: R$ 351.76


,Data,Preco_Arroba,Variacao_Diaria
0,2016-01-02,153.00,2016
1,2016-01-03,155.85,2016
2,2016-01-04,157.48,2016
3,2016-01-06,156.50,2016
4,2016-01-07,158.55,2016


In [ ]:
# Estatísticas básicas
print("📊 Estatísticas do preço da arroba (Cepea):")
display(cepea['Preco_Arroba'].describe())

# Preço médio por ano
cepea['Ano'] = cepea['Data'].dt.year
media_anual = cepea.groupby('Ano')['Preco_Arroba'].mean().round(2)
print("\nMédia anual:")
display(media_anual)

📊 Estatísticas do preço da arroba (Cepea):


count    989.000000
mean     232.763074
std       74.197170
min      124.790000
25%      152.290000
50%      235.910000
75%      309.460000
max      357.410000
Name: Preco_Arroba, dtype: float64


Média anual:


Ano
2016    153.86
2017    139.80
2018    145.77
2019    161.31
2020    225.93
2021    304.71
2022    321.62
2023    259.43
2024    256.90
2025    318.75
2026    339.52
Name: Preco_Arroba, dtype: float64

In [ ]:
# Gráfico 1: Evolução do preço da arroba 
fig = px.line(cepea, 
              x='Data', 
              y='Preco_Arroba',
              title='📈 Evolução Histórica do Preço do Boi Gordo (Cepea/ESALQ)',
              labels={'Preco_Arroba': 'Preço por Arroba (R$)', 'Data': 'Data'},
              template='plotly_white')

fig.update_traces(line=dict(width=2.5))
fig.update_layout(hovermode='x unified', height=600)
fig.show()

In [ ]:
# Gráfico 2: Preço médio por mês 
cepea['Mes'] = cepea['Data'].dt.month_name(locale='pt_BR')
meses_ordem = ['Janeiro','Fevereiro','Março','Abril','Maio','Junho',
               'Julho','Agosto','Setembro','Outubro','Novembro','Dezembro']

sazonal = cepea.groupby('Mes')['Preco_Arroba'].mean().reindex(meses_ordem)

fig2 = px.bar(sazonal, 
              title='📅 Sazonalidade: Preço médio por mês do ano',
              labels={'value': 'Preço médio (R$/arroba)', 'Mes': 'Mês'},
              template='plotly_white')
fig2.show()

In [ ]:
# Calculadora de exemplo
def calcular_valor_animal(peso_vivo_kg, preco_arroba_atual):
    arrobas = peso_vivo_kg / 30
    valor_total = arrobas * preco_arroba_atual
    return arrobas, round(valor_total, 2)

# Exemplo prático com preço atual 
preco_atual = cepea['Preco_Arroba'].iloc[-1]
print(f"💰 Preço atual da arroba: R$ {preco_atual:.2f}")

arrobas, valor = calcular_valor_animal(510, preco_atual)
print(f"🐂 Boi de 510 kg = {arrobas:.1f} arrobas → Valor estimado: R$ {valor}")

💰 Preço atual da arroba: R$ 351.76
🐂 Boi de 510 kg = 17.0 arrobas → Valor estimado: R$ 5979.92
